In [9]:
# Define parameters
keyword = '/g/11c265kd70'
gprop = 'youtube'
geo = 'IN'
timeframe = '2025-06-15 2025-12-29'
params = {
    "api_key" : "b9470e54fd35b3f2174c686b526660af023bbcb1521aa1211194bef5a5a663c4",
    "engine": "google_trends",
    "q": keyword,
    "geo": geo,
    "date": timeframe,
    "data_type": "TIMESERIES",
    "tz": "330",
    "gprop": gprop if gprop != "web" else ""
}

In [ ]:
from datetime import datetime, timedelta
from statistics import mean
import gtrends.db.utils as db_utils
from gtrends.db.keywords import GoogleTrendsRepository

class GoogleTrendSyncer:
    """
    Handles normalization and continuity of Google Trends time series
    across overlapping fetch windows.
     - Feeds it into the database while also updating the past entries
    """

    DATE_FMT = "%d-%m-%Y"

    def __init__(self):
        self.gtr = GoogleTrendsRepository()
    
    def _find_overlap(self, past, new):
        past_map = {p["date"]: p["value"] for p in past}
        overlap = []

        for row in new:
            if row["date"] in past_map:
                overlap.append((past_map[row["date"]], row["value"]))

        return overlap
    
    def compute_scaling_factor(self, overlap):
        """
        Scaling factor = mean(past_values) / mean(new_values)
        """
        if len(overlap) < 1:
            return 1.0  # insufficient overlap → no scaling

        past_vals = [p for p, _ in overlap]
        new_vals = [n for _, n in overlap]

        if mean(new_vals) == 0:
            return 1.0

        return mean(past_vals) / mean(new_vals)
    
    def normalize_new_data(self, new_data, scale):
        normalized = []

        for row in new_data:
            normalized.append({
                "date": row["date"],
                "value": round(row["value"] * scale, 2)
            })

        return normalized
    
    def merge_timeseries(self, past, normalized_new):
        merged = {row["date"]: row["value"] for row in past}

        for row in normalized_new:
            merged[row["date"]] = row["value"]

        return [
            {"date": d, "value": v}
            for d, v in sorted(merged.items(), key=lambda x: datetime.strptime(x[0], self.DATE_FMT))
        ]
    
    def upsert_timeseries(self, keyword, geo, data):
        query = """
        INSERT INTO google_trends_timeseries (keyword, geo, date, value)
        VALUES (%s, %s, %s, %s)
        ON DUPLICATE KEY UPDATE value = VALUES(value)
        """

        params = [
            (
                keyword,
                geo,
                datetime.strptime(row["date"], self.DATE_FMT),
                row["value"]
            )
            for row in data
        ]

        self.db.executemany(query, params)
        self.db.commit()

    def generate_windows(self, start, end, window_days = 240, overlap_days = 30):
        w_end = end

        windows = []
        while w_end > start:
            w_start = max(w_end - timedelta(days = window_days), start) # enforces 30 days overlap
            windows.append((w_start.isoformat(), w_end.isoformat()))
            w_end = w_end - timedelta(days = window_days - overlap_days)  # shift left by 210 days

        return windows[::-1]

In [ ]:
gts = GoogleTrendSyncer()

def sync(search_id):
    """
    fetch_fn(start, end) → returns list of {date, value}
    """
    def get_windows(search_id):
        past = gts.gtr.get_past_data(search_id)
        if past:
            start = datetime.fromtimestamp(past[-1].get('timestamp')) - timedelta(days=30)
        else:
            start = datetime.now().date() - timedelta(days = 4 * 365)
        end = datetime.now().date()

        windows = gts.generate_windows(start, end)
        return windows
    
    windows = get_windows(search_id)
    stitched = []

    for start, end in windows:
        chunk = fetch_fn(start, end)

        if not stitched:
            stitched = chunk
            continue

        overlap = self._find_overlap(stitched, chunk)
        scale = self.compute_scaling_factor(overlap)
        chunk = self.normalize_new_data(chunk, scale)
        stitched = self.merge_timeseries(stitched, chunk)

    self.upsert_timeseries(keyword, geo, stitched)

In [ ]:
from gtrends.chunk.fetch import SerpApiTrendsFetcher
stf = SerpApiTrendsFetcher()
stf.initialize()

gtr = GoogleTrendsRepository()
searches_pending = gtr.searches_to_update()

current_search = searches_pending[0]
windows = sync(current_search.get('id'))

start, end = windows[0]
chunk = stf.fetch_chunk(keyword = current_search.get('search_keyword')
                        start = start, end)

In [26]:
import os, sys
sys.path.insert(0, ".")
os.environ["SERPAPI_KEYS"] = "47cfd4d4c42c64c4050c883f7e8d43f93c63027b2d89fbc1f5b95ea9bbb801c2"


In [ ]:
from gtrends.chunk.fetch import SerpApiTrendsFetcher

fetcher = SerpApiTrendsFetcher(geo="IN")

try:
    fetcher.initialize()
    data = fetcher.fetch_chunk(
    keyword="/g/11c265kd70",
    start="2024-01-01",
    end="2024-12-31",
    gprop="youtube"
    )
    print(f"success")
except Exception as e:
    print(f"error:{e}")

In [24]:
import os,time
from datetime import datetime
from chunk.fetch import SerpApiTrendsFetcher
from process.sync import GoogleTrendSyncer

fetcher = SerpApiTrendsFetcher()
syncer = GoogleTrendSyncer()

Api_key = "47cfd4d4c42c64c4050c883f7e8d43f93c63027b2d89fbc1f5b95ea9bbb801c2"

kw_to_search = "Holi"
window_days = 240
overlap_days = 30

# start = datetime.strptime("02-02-2021", "%d-%m-%Y")
# # start = datetime.today().strftime("%d-%m-%Y") - "02-02-2021"
# end = datetime.today().strftime("%d-%m-%Y")

start = datetime.strptime("02-02-2021", "%d-%m-%Y")
end = datetime.today()

try:
    windows = syncer.generate_windows(start,end,window_days,overlap_days)
except Exception as e:
    print(f"error generating window: {e}")


for window in windows:

    try:
        chunk = fetcher.fetch_chunk(kw_to_search,start,end,"youtube")
        print(f"chunk fetched sucessfully")
    except Exception as e:
        print(f"error fetching {e}")

    print(f"length of chunk: {len(chunk)}")

    time.sleep(3)
    with open("data.csv",'a') as af:
        af.append(chunk)




error fetching 'SerpApiTrendsFetcher' object has no attribute 'geo'


NameError: name 'chunk' is not defined

In [ ]:
import os,time
from datetime import datetime, timedelta
import csv
from chunk.fetch import SerpApiTrendsFetcher
from process.sync import GoogleTrendSyncer


syncer = GoogleTrendSyncer()

Api_key = "dd91f8ff20268440571b3410cec679caacbaa390ef89e238e20d5d649829f35d"
fetcher = SerpApiTrendsFetcher(Api_key)

# kw_to_search = "Rajasthan"
search_list = ["Jaipur","NEET"]
window_days = 240
overlap_days = 30

start = datetime.strptime("2021-04-02", "%Y-%m-%d").strftime("%Y-%m-%d")
end = datetime.today().strftime("%Y-%m-%d")

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
DATA_FOLDER = os.path.join(BASE_DIR, "Data")

# Create the 'data' folder if it doesn't exist
if not os.path.exists(DATA_FOLDER):
     os.makedirs(DATA_FOLDER)

for search in search_list:

        safe_keyword = "".join(c if c.isalnum() or c in ("_", "-") else "_" for c in search)

        csv_file = os.path.join(DATA_FOLDER, f"{safe_keyword}.csv")

        file_exists = os.path.isfile(csv_file)

        # get last date from csv
        last_date = syncer.get_last_date(csv_file,"date")
        if last_date:
            print(f"fetching from the date:{last_date}")

            start_dt = (last_date - timedelta(days=overlap_days)).strftime("%Y-%m-%d")

        # generate windows
        try:
            windows = syncer.generate_windows(start_dt,end,window_days,overlap_days)
            print(f"Windows generated of length:{len(windows)}")
        except Exception as e:
            print(f"error generating window: {e}")


        for window in windows:

            w_start, w_end = window
            print(f"fetching window :{w_start}->{w_end}")

            # fetch data
            try:
                chunk = fetcher.fetch_chunk(search,w_start,w_end,"youtube")
                print(f"chunk fetched sucessfully")
            except Exception as e:
                print(f"error fetching {e}")

            
            # get last 30 days data from the csv file to find overlap
            if file_exists:
                with open(csv_file, mode="r",newline="", encoding="utf-8") as f:
                    reader = list(csv.reader(f))

                    rows = reader[1:]           # actual data rows

                    past = rows[-30:] 
                    if not past:
                        past = chunk

                overlap = syncer.find_overlap(past, chunk)
                scale = syncer.compute_scaling_factor(overlap)
            
            try:
                chunk = syncer.normalize_new_data(chunk, scale)
            except Exception as e:
                print(f"Normalization error:{e}")
                continue


            if file_exists and rows:
                last_saved_date = rows[-1][0] 
                # Filter chunk to keep only dates after the last saved date
                chunk = [row for row in chunk if row["date"] > last_saved_date]

            # append the data in file
            if chunk: # Only write if there is data left
                with open(csv_file, mode="a", newline="", encoding="utf-8") as f:
                    writer = csv.writer(f)

                    if not file_exists:
                        writer.writerow(["date", search])
                        file_exists = True

                    for row in chunk:
                        writer.writerow([
                            row["date"],
                            row["value"]
                        ])
                print(f"Data saved to {csv_file}")
            else:
                print("No new data to save for this window.")




